# 🎓 AegisX — Stage 2 SFT on Kaggle (free GPU)

Fine-tunes **YOUR OWN** pre-trained model (from `aegisx_train_kaggle.ipynb`)
on the 998 bilingual instruction rows. Pure zero - no Qwen, no third-party
base model.

**How the stage-1 model gets here (Kaggle has no Drive):**
1. After pre-training, you downloaded `aegisx-mini.zip`
2. On Kaggle: **Add input → New dataset → Create a new dataset** → upload
   `model.pt`, `tokenizer.json`, `config.json` (from the ZIP)
3. In this notebook: **Add input** → select that dataset → it appears at
   `/kaggle/input/<dataset-name>/`

**Setup:** Settings → Accelerator **GPU P100** + Internet **ON**.

## 1. Setup

In [ ]:
!pip install -q torch

import os
import torch
from pathlib import Path
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Locate your stage-1 checkpoint in /kaggle/input

In [ ]:
# Auto-find model.pt under /kaggle/input (your uploaded dataset).
import os
from pathlib import Path

candidates = list(Path('/kaggle/input').rglob('model.pt')) if Path('/kaggle/input').exists() else []
if candidates:
    INIT_MODEL = str(candidates[0])
    print('Found checkpoint:', INIT_MODEL)
else:
    INIT_MODEL = input('model.pt not found in /kaggle/input. Paste its full path (e.g. /kaggle/input/mymodel/model.pt): ').strip()
assert os.path.exists(INIT_MODEL), f'Not found: {INIT_MODEL} - add your dataset via the right panel (Add input)'
print('init from:', INIT_MODEL)

## 3. Get the AegisX code + instruction data

In [ ]:
WORK = '/kaggle/working/aegisx'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
if not os.path.isdir('.git'):
    !git clone https://github.com/FerzDevZ/AegisX.git .
else:
    !git -C . pull --ff-only
print('Repo ready.')

In [ ]:
# Build the SFT chat text from data/finetune/instructions.jsonl (998 rows).
!python scripts/build_sft_text.py --out data/finetune/sft_chat.txt

SFT_DATA = '/kaggle/working/aegisx-sft-data'
os.makedirs(SFT_DATA, exist_ok=True)
!cp data/finetune/sft_chat.txt {SFT_DATA}/sft.txt
print('SFT data ready at', SFT_DATA)

## 4. SFT train (your own model)

In [ ]:
MAX_STEPS  = 1500
BATCH_SIZE = 16
GRAD_ACCUM = 4
EVAL_EVERY = 250
EARLY_STOP = 4
LR         = 1e-4   # gentle: don't destroy stage-1 knowledge
WARMUP     = 100
SFT_OUT    = '/kaggle/working/checkpoints/aegisx-sft'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
!python -m aegisx.train \
    --data {SFT_DATA} \
    --out {SFT_OUT} \
    --init-from {INIT_MODEL} \
    --max-steps {MAX_STEPS} \
    --batch-size {BATCH_SIZE} \
    --grad-accum {GRAD_ACCUM} \
    --eval-every {EVAL_EVERY} \
    --early-stop-patience {EARLY_STOP} \
    --lr {LR} \
    --warmup-steps {WARMUP} \
    --device {DEVICE}

## 5. Compare: before vs after SFT

In [ ]:
import os
tok = f'{SFT_OUT}/tokenizer.json'
if os.path.exists(f'{SFT_OUT}/model.pt'):
    if os.path.exists(INIT_MODEL):
        print('--- BEFORE (stage-1 model) ---')
        !python -m aegisx.eval --model {INIT_MODEL} --tokenizer {tok} --device {DEVICE} --max-new-tokens 80 2>/dev/null | tail -4
    print('--- AFTER (SFT model) ---')
    !python -m aegisx.eval --model {SFT_OUT}/model.pt --tokenizer {tok} --device {DEVICE} --max-new-tokens 80 2>/dev/null | tail -4
else:
    print('Skipped: SFT model not found - check the training cell.')

## 6. Chat with the SFT model

In [ ]:
import os
if os.path.exists(f'{SFT_OUT}/model.pt'):
    !python -m aegisx.chat --model {SFT_OUT}/model.pt --tokenizer {SFT_OUT}/tokenizer.json \
        --prompt "You are AegisX, a cybersecurity assistant. User: Apa itu SQL injection?\n\nAegisX:" \
        --max-new-tokens 150 --temperature 0.7 --top-k 50
else:
    print('Skipped: SFT model not found.')

## 7. Package for manual Hugging Face upload

In [ ]:
import os, shutil, zipfile
from pathlib import Path

if os.path.exists(f'{SFT_OUT}/model.pt'):
    EXPORT_DIR = Path('/kaggle/working/export/aegisx-mini-sft')
    if EXPORT_DIR.exists():
        shutil.rmtree(EXPORT_DIR)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy(f'{SFT_OUT}/model.pt', EXPORT_DIR / 'model.pt')
    shutil.copy(f'{SFT_OUT}/tokenizer.json', EXPORT_DIR / 'tokenizer.json')
    shutil.copy(f'{SFT_OUT}/config.json', EXPORT_DIR / 'config.json')

    card = Path('hf/MODEL_CARD.md')
    if card.exists():
        shutil.copy(card, EXPORT_DIR / 'README.md')

    KNOW = EXPORT_DIR / 'knowledge'
    KNOW.mkdir(exist_ok=True)
    for f in sorted(Path('data/raw').glob('*.txt')):
        shutil.copy(f, KNOW / f.name)

    zip_path = Path(str(EXPORT_DIR) + '.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(EXPORT_DIR.iterdir()):
            if f.is_dir():
                for inner in f.rglob('*'):
                    if inner.is_file():
                        zf.write(inner, arcname=f'{f.name}/{inner.name}')
            else:
                zf.write(f, arcname=f.name)
    print('Export folder:')
    for f in sorted(EXPORT_DIR.iterdir()):
        print(f'  {f.name}')
    print(f'ZIP: {zip_path}')
else:
    print('Skipped: SFT model not found.')

## 8. ⚠️ DOWNLOAD the export NOW

Kaggle sessions end without warning. Click the link and save the ZIP, then
upload it manually to Hugging Face.

In [ ]:
import os
zip_path = '/kaggle/working/export/aegisx-mini-sft.zip'
if os.path.exists(zip_path):
    print('ZIP ready. Click the link below to download:')
    from IPython.display import FileLink, display
    display(FileLink(zip_path))
else:
    print('ZIP not found - run cell 7 first.')